In [1]:
!rm -rf /kaggle/working/*

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

In [3]:
train=pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/train.csv")

test=pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/test.csv")

sub=pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/sample_submission.csv")

In [4]:
train.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [5]:
train.shape

(439140, 16)

In [6]:
train.isnull().sum()

id                        0
Driver                    0
Compound                  0
Race                      0
Year                      0
PitStop                   0
LapNumber                 0
Stint                     0
TyreLife                  0
Position                  0
LapTime (s)               0
LapTime_Delta             0
Cumulative_Degradation    0
RaceProgress              0
Position_Change           0
PitNextLap                0
dtype: int64

In [7]:
test.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
0,439140,D119,MEDIUM,British Grand Prix,2023,0,21,1,21.0,4,93.387,0.280,-4.984,0.403846,0.0
1,439141,VER,MEDIUM,Abu Dhabi Grand Prix,2023,0,24,1,24.0,1,90.867,-0.129,-1.990,0.413793,0.0
2,439142,D270,MEDIUM,British Grand Prix,2023,0,24,1,24.0,11,92.871,0.041,-8.842,0.461538,0.0
3,439143,D112,SOFT,São Paulo Grand Prix,2024,0,6,2,4.0,15,94.967,-19.741,8.250,0.077922,1.0
4,439144,AND,HARD,United States Grand Prix,2024,0,52,2,29.0,12,99.112,0.930,-20.848,0.722222,7.0


In [8]:
test.shape

(188165, 15)

In [9]:
test.isnull().sum()

id                        0
Driver                    0
Compound                  0
Race                      0
Year                      0
PitStop                   0
LapNumber                 0
Stint                     0
TyreLife                  0
Position                  0
LapTime (s)               0
LapTime_Delta             0
Cumulative_Degradation    0
RaceProgress              0
Position_Change           0
dtype: int64

In [10]:
sub.head()

,id,PitNextLap
0,439140,0
1,439141,0
2,439142,0
3,439143,0
4,439144,0


In [11]:
cat_cols = ['Driver', 'Compound', 'Race']

for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))
    test[col] = test[col].map(lambda s: '<unknown>' if s not in le.classes_ else s)
    le.classes_ = np.append(le.classes_, '<unknown>')
    test[col] = le.transform(test[col].astype(str))

X = train.drop(['id', 'PitNextLap'], axis=1)
y = train['PitNextLap']
X_test = test.drop(['id'], axis=1)

In [12]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lgb_test_preds = np.zeros(len(X_test))
lgb_oof = np.zeros(len(X))
xgb_test_preds = np.zeros(len(X_test))
xgb_oof = np.zeros(len(X))
cb_test_preds = np.zeros(len(X_test))
cb_oof = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    model_lgb = lgb.LGBMClassifier(n_estimators=1000, random_state=42, learning_rate=0.05, max_depth=6)
    model_lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)])
    lgb_oof[val_idx] = model_lgb.predict_proba(X_val)[:, 1]
    lgb_test_preds += model_lgb.predict_proba(X_test)[:, 1] / kf.n_splits
    
    model_xgb = xgb.XGBClassifier(n_estimators=1000, random_state=42, learning_rate=0.05, max_depth=6, eval_metric="auc")
    model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    xgb_oof[val_idx] = model_xgb.predict_proba(X_val)[:, 1]
    xgb_test_preds += model_xgb.predict_proba(X_test)[:, 1] / kf.n_splits
    
    model_cb = CatBoostClassifier(iterations=1000, random_state=42, learning_rate=0.05, depth=6, eval_metric="AUC", verbose=False)
    model_cb.fit(X_train, y_train, eval_set=[(X_val, y_val)])
    cb_oof[val_idx] = model_cb.predict_proba(X_val)[:, 1]
    cb_test_preds += model_cb.predict_proba(X_test)[:, 1] / kf.n_splits
    
    print(f"Finished Fold {fold + 1} / {kf.n_splits}")

print(roc_auc_score(y, lgb_oof))
print(roc_auc_score(y, xgb_oof))
print(roc_auc_score(y, cb_oof))

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 69904, number of negative: 281408
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008855 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1519
[LightGBM] [Info] Number of data points in the train set: 351312, number of used features: 14
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.198980 -> initscore=-1.392683
[LightGBM] [Info] Start training from score -1.392683
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

In [13]:
final_preds = (lgb_test_preds + xgb_test_preds + cb_test_preds) / 3

sub['PitNextLap'] = final_preds
sub.to_csv('submission.csv', index=False)
sub.head()

,id,PitNextLap
0,439140,0.004772
1,439141,0.004425
2,439142,0.004116
3,439143,0.129787
4,439144,0.835867
